# Acquisition notebook

Communication with PM100A power-meter

In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import pyvisa as pv
import serial
import serial.tools.list_ports

In [3]:
from utils.pm100a import record_pm100a
from utils.oc3 import OC3
from utils.naming import make_run_dir

In [4]:
import labmate
from labmate.acquisition_notebook import AcquisitionAnalysisManager

from datetime import datetime
from pathlib import Path

Set directory for data collection

In [5]:
# DATA_DIR = "data/data_pm100a"
DATA_DIR = "data/data_powermeter"
os.makedirs(DATA_DIR, exist_ok=True)

Check for PM100A and PM100D (pesto)

In [6]:
# Check available VISA resources
rm = pv.ResourceManager()
print(rm.list_resources())

('USB0::0x0699::0x039F::C010359::INSTR', 'USB0::0x1313::0x8079::P1007388::INSTR', 'USB0::0x1313::0x8078::P0025589::INSTR', 'ASRL1::INSTR')


In [7]:
# Connect to PM100A power meter via PyVISA
# you can use the tool Power Meter Driver Switcher to switch between the two drivers, the PM100D.dll driver and the new TLPM.dll driver. 
# Pyvisa may not recognize the PM100A with the WinUSB driver, so you may need to switch to the Visa driver. 
pm100a = rm.open_resource('USB0::0x1313::0x8079::P1007388::INSTR')  # PANDA powermeter
# pm100d = rm.open_resource('USB0::0x1313::0x8078::P0008159::INSTR')  # PESTO powermeter
pm100d = rm.open_resource('USB0::0x1313::0x8078::P0025589::INSTR')  # Musiqs powermeter
print(pm100a.query('*IDN?'))
print(pm100d.query('*IDN?'))

Thorlabs,PM100A,P1007388,2.5.0

Thorlabs,PM100D,P0025589,2.8.1



In [26]:
plist = list(serial.tools.list_ports.comports())
print(plist)

[<serial.tools.list_ports_common.ListPortInfo object at 0x000001B888A7AEA0>, <serial.tools.list_ports_common.ListPortInfo object at 0x000001B888A7AD70>]


In [27]:
# Connect to OC3 temperature controller via pyserial
# OC_PORT = "COM3" 
OC_PORT = "COM5"
oc3 = OC3(port=OC_PORT)
print("Connected to OC3 on port", OC_PORT, oc3.status())

Connected to OC3 on port COM5 b'j5543.050;43.050;1;03.5;0;0;1;23.973;c24/1.54p;0;0;43.050;65'


In [12]:
# Record parameters
PM_SAMPLE_DELAY = 0.1 # seconds
PM_DURATION = 5 # seconds

### Reflectivity and Transmitivity measurements

In [133]:
# Set names
# SAMPLE = "laseroptik-T004Q2"

# SAMPLE = "metallic_unknown"
# --- Cavity Mirrors --- #
# SAMPLE = "laseroptic_panda_hr"
# SAMPLE = "laseroptic_panda_roc_150"
# SAMPLE = "laseroptic_panda_coupler_5"
# MEAS = "reflectivity"

# --- EQ15B ---#
MEAS = "transmitivity"
SAMPLE = "eq15b_flat_HR426_1inch_face2"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

Saving data to: data/data_pm100a\transmitivity\2026-06-01_eq15b_flat_HR426_1inch_face2_001


In [134]:
ACQ_CELL = f"{MEAS}_{SAMPLE}"
print(ACQ_CELL)

transmitivity_eq15b_flat_HR426_1inch_face2


In [ ]:
# Wavelength
wavelength = 780 # nm

# Acquisition and analysis
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

mean_mW, error_mW = record_pm100a(
        pm100a,
        duration_s=2,
        dt_s=PM_SAMPLE_DELAY
    )

print(f"Mean power = {mean_mW:.6f} ± {error_mW:.6f} mW")

aqm.save_acquisition(mean_mW=mean_mW, error_mW=error_mW, wavelength=wavelength, sample=SAMPLE)

INFO:1:2026_06_01__15_42_57__transmitivity_eq15b_flat_HR426_1inch_face2


Mean power = 22.476131 ± 0.032683 mW


### Basic recording

PM100A records SHG output
PM100D records 780nm input

In [13]:
MEAS = "power_scan_shg"
# SAMPLE = "ppktp_svenska_10"
SAMPLE = "ppktp_svenska_20_jesse"

In [14]:
RUN_DIR = r'data/data_powermeter/power_scan_shg/2026-06-12_ppktp_svenska_20_jesse_001'

In [20]:
RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

Directory 2026-06-12_ppktp_svenska_20_jesse_001 already exists. Incrementing index.
Saving data to: data/data_powermeter\power_scan_shg\2026-06-12_ppktp_svenska_20_jesse_002


In [21]:
ACQ_CELL = f"{MEAS}_{SAMPLE}"
print(ACQ_CELL)

power_scan_shg_ppktp_svenska_20_jesse


In [46]:
# Wavelength
wavelength = 390 # nm
head = "pm100a"

# Acquisition and analysis
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

mean_mW, error_mW = record_pm100a(
        pm100a,
        duration_s=2,
        dt_s=PM_SAMPLE_DELAY
    )

Tmeas, Terror = oc3.get_mean_temperature(duration_s=4.0)
print(f"Current T: {Tmeas:.3f}  ± {Terror:.3f} °C")
print(f"Mean power = {mean_mW:.6f} ± {error_mW:.6f} mW")

aqm.save_acquisition(mean_mW=mean_mW, error_mW=error_mW, Tmeas=Tmeas, Terror=Terror, wavelength=wavelength, sample=SAMPLE, head=head)

INFO:1:2026_06_12__16_37_37__power_scan_shg_ppktp_svenska_20_jesse


Current T: 42.718  ± 0.001 °C
Mean power = 21.052122 ± 0.055888 mW


In [45]:
# Wavelength
wavelength = 780 # nm
head = "pm100d"

# Acquisition and analysis
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

mean_mW, error_mW = record_pm100a(
        pm100d,
        duration_s=2,
        dt_s=PM_SAMPLE_DELAY
    )

print(f"Mean power = {mean_mW:.6f} ± {error_mW:.6f} mW")

aqm.save_acquisition(mean_mW=mean_mW, error_mW=error_mW, wavelength=wavelength, sample=SAMPLE, head=head)

INFO:1:2026_06_12__16_35_42__power_scan_shg_ppktp_svenska_20_jesse


Mean power = 142.821685 ± 0.034678 mW
